<a href="https://colab.research.google.com/github/omonigho-egbo-15/MidTerm_EDA_GP_1_Project/blob/main/Midterm_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **MIDTERM_EDA_GROUP_PROJECT**
## 1. ***Problem Statement: Student Dropout Prediction***

This project aims to build a machine learning model that predict student dropout using the kaggle dataset ('student_dropout_dataset_v3.csv'). The goal is to develop a **binary classification model** that can identify students at risk of dropping out, allowing educational institutions to implement timely interventions.

*   **Target Variable (`y`)**: `Dropout` (0 = Retained, 1 = Dropped Out)
*   **Features (`X`)**: All other relevant columns, excluding `Student_ID` which is a unique identifier and not predictive.

The process will involve data loading, initial inspection, splitting data into training and testing sets, comprehensive data preprocessing (handling missing values, encoding categorical features, feature engineering, and scaling), and finally, training and evaluating a baseline Logistic Regression model.

## 2. ***Load Dataset and Initial Inspection***

We started by loading the dataset and performing an initial check to understand its structure, identify data types, and spot any immediate issues like missing values. We then immediately drop the `Student_ID` column as it is an identifier and holds no predictive power for the model, thus avoiding any potential data leakage or overfitting from this column.

In [9]:
import pandas as pd
import numpy as np
from google.colab import files

# Load the dataset from the uploaded file
# If not found, it will prompt you for upload.
try:
    df = pd.read_csv('/content/student_dropout_dataset_v3.csv')
    print("Dataset loaded successfully from /content/student_dropout_dataset_v3.csv")
except FileNotFoundError:
    print("Dataset not found at '/content/student_dropout_dataset_v3.csv'.")
    print("Please upload the 'student_dropout_dataset_v3.csv' file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
        df = pd.read_csv(f'/content/{file_name}')
        print(f"Dataset '{file_name}' loaded successfully.")
    else:
        raise FileNotFoundError("No file uploaded. Please upload 'student_dropout_dataset_v3.csv' to proceed.")

print("--- Initial DataFrame Head ---")
display(df.head())

# Drop 'Student_ID' as it's a unique identifier and not a predictive feature
df = df.drop(columns=['Student_ID'])
print("\n--- DataFrame Head after dropping Student_ID ---")
display(df.head())

print("\n--- DataFrame Info (before any cleanup) ---")
df.info()

print("\n--- Missing Values Count (before any cleanup) ---")
print(df.isnull().sum())

Dataset loaded successfully from /content/student_dropout_dataset_v3.csv
--- Initial DataFrame Head ---


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0



--- DataFrame Head after dropping Student_ID ---


,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0



--- DataFrame Info (before any cleanup) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    10000 non-null  float64
 1   Gender                 10000 non-null  object 
 2   Family_Income          9500 non-null   float64
 3   Internet_Access        10000 non-null  object 
 4   Study_Hours_per_Day    9500 non-null   float64
 5   Attendance_Rate        10000 non-null  float64
 6   Assignment_Delay_Days  10000 non-null  int64  
 7   Travel_Time_Minutes    10000 non-null  float64
 8   Part_Time_Job          10000 non-null  object 
 9   Scholarship            10000 non-null  object 
 10  Stress_Index           9500 non-null   float64
 11  GPA                    10000 non-null  float64
 12  Semester_GPA           10000 non-null  float64
 13  CGPA                   10000 non-null  float64
 14  Semester  

## 3. Data Splitting: Training and Testing Sets

Before performing any preprocessing, we split the data into training (70%) and testing (30%) sets. This is crucial to prevent **data leakage**, where information from the test set could inadvertently influence the training process, leading to overly optimistic model performance. All subsequent cleaning and preprocessing steps will be fitted *only* on the training data and then applied to both sets.

In [10]:
from sklearn.model_selection import train_test_split

# Define Features (X) and Target (y)
# 'Dropout' is our label (0 = Retained, 1 = Dropped out)
X = df.drop(columns=['Dropout'])
y = df['Dropout']

# Split data into training and testing sets (70% train, 30% test)
# stratify=y ensures that the proportion of 'Dropout' cases is the same in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

print("\n--- X_train Data Types Before Cleanup ---")
print(X_train.dtypes)

X_train shape: (7000, 17)
y_train shape: (7000,)
X_test shape: (3000, 17)
y_test shape: (3000,)

--- X_train Data Types Before Cleanup ---
Age                      float64
Gender                    object
Family_Income            float64
Internet_Access           object
Study_Hours_per_Day      float64
Attendance_Rate          float64
Assignment_Delay_Days      int64
Travel_Time_Minutes      float64
Part_Time_Job             object
Scholarship               object
Stress_Index             float64
GPA                      float64
Semester_GPA             float64
CGPA                     float64
Semester                  object
Department                object
Parental_Education        object
dtype: object


## 4. ***Data Before Cleanup (Training Data)***

Let's take a closer look at the training data before applying any cleaning or preprocessing steps. This helps us confirm the initial state and understand what needs to be done.

In [11]:
print("--- X_train Head Before Cleanup ---")
display(X_train.head())

print("\n--- X_train Info Before Cleanup ---")
X_train.info()

print("\n--- Missing Values in X_train Before Cleanup ---")
print(X_train.isnull().sum())

--- X_train Head Before Cleanup ---


,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education
9252,20.0,Male,25000.0,Yes,3.29,65.9,4,34.9,Yes,Yes,4.5,1.90,1.86,1.95,Year 4,Arts,Bachelor
598,20.0,Male,25000.0,Yes,3.16,79.2,4,21.9,Yes,No,2.8,3.47,3.61,3.62,Year 3,Engineering,Master
7394,21.1,Female,34229.0,Yes,5.94,62.3,2,15.5,No,No,7.9,2.02,2.46,2.47,Year 2,Engineering,Master
5027,20.0,Female,48446.0,Yes,5.26,100.0,2,23.3,No,Yes,6.0,1.41,1.33,1.39,Year 4,Arts,PhD
1396,18.0,Female,25000.0,No,2.28,96.3,0,22.3,No,No,5.4,3.78,3.98,3.98,Year 1,Business,High School



--- X_train Info Before Cleanup ---
<class 'pandas.core.frame.DataFrame'>
Index: 7000 entries, 9252 to 6816
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    7000 non-null   float64
 1   Gender                 7000 non-null   object 
 2   Family_Income          6646 non-null   float64
 3   Internet_Access        7000 non-null   object 
 4   Study_Hours_per_Day    6646 non-null   float64
 5   Attendance_Rate        7000 non-null   float64
 6   Assignment_Delay_Days  7000 non-null   int64  
 7   Travel_Time_Minutes    7000 non-null   float64
 8   Part_Time_Job          7000 non-null   object 
 9   Scholarship            7000 non-null   object 
 10  Stress_Index           6646 non-null   float64
 11  GPA                    7000 non-null   float64
 12  Semester_GPA           7000 non-null   float64
 13  CGPA                   7000 non-null   float64
 14  Semester             

## 5. ***Data Cleaning and Preprocessing Pipeline***

Now, we'll build a robust preprocessing pipeline using `sklearn.pipeline.Pipeline` and `sklearn.compose.ColumnTransformer`. This ensures that all transformations are applied consistently and correctly, especially between training and testing data.

The steps are:

1.  **Imputation**: Fill missing values.
    *   Numerical columns (`Family_Income`, `Study_Hours_per_Day`, `Stress_Index`): Use the **median** strategy to be robust to outliers.
    *   Categorical columns (`Parental_Education`): Use the **most frequent** strategy.
2.  **Feature Engineering**: Create a new, potentially more informative feature.
    *   We'll create `CGPA_Per_Attendance` by dividing `CGPA` by `Attendance_Rate`. This might capture a student's academic efficiency relative to their presence.
3.  **Encoding**: Convert categorical text data into numerical format.
    *   `OneHotEncoder` for nominal categorical features (e.g., `Gender`, `Department`). This avoids implying ordinal relationships where none exist.
4.  **Scaling**: Standardize numerical features.
    *   `StandardScaler` to transform features to have a mean of 0 and standard deviation of 1. This is important for many machine learning algorithms (like Logistic Regression) that are sensitive to feature scales.

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np # Import numpy for np.number

# Identify numerical and categorical columns for the pipeline
# Exclude 'CGPA_Per_Attendance' for now as it will be engineered later
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# --- Step 1: Imputation (Custom step before ColumnTransformer for feature engineering) ---
# For numerical features, use median imputation
numerical_imputer = SimpleImputer(strategy='median')

# For categorical features, use most frequent(mode) imputation
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Apply imputation separately to X_train and X_test to keep DataFrame structure for feature engineering
X_train_imputed_num = pd.DataFrame(numerical_imputer.fit_transform(X_train[numerical_cols]),
                                   columns=numerical_cols, index=X_train.index)
X_train_imputed_cat = pd.DataFrame(categorical_imputer.fit_transform(X_train[categorical_cols]),
                                   columns=categorical_cols, index=X_train.index)
X_train_imputed = pd.concat([X_train_imputed_num, X_train_imputed_cat], axis=1)

X_test_imputed_num = pd.DataFrame(numerical_imputer.transform(X_test[numerical_cols]),
                                  columns=numerical_cols, index=X_test.index)
X_test_imputed_cat = pd.DataFrame(categorical_imputer.transform(X_test[categorical_cols]),
                                  columns=categorical_cols, index=X_test.index)
X_test_imputed = pd.concat([X_test_imputed_num, X_test_imputed_cat], axis=1)

print("--- Missing values after imputation in X_train (should be 0) ---")
print(X_train_imputed.isnull().sum().sum())
print("--- Missing values after imputation in X_test (should be 0) ---")
print(X_test_imputed.isnull().sum().sum())

# --- Step 2: Feature Engineering ---
# This creates a new feature: 'CGPA_Per_Attendance'
# Add a small epsilon to Attendance_Rate to prevent division by zero for students with 0% attendance
X_train_fe = X_train_imputed.copy()
X_test_fe = X_test_imputed.copy()

X_train_fe['CGPA_Per_Attendance'] = X_train_fe['CGPA'] / (X_train_fe['Attendance_Rate'] + 1e-6)
X_test_fe['CGPA_Per_Attendance'] = X_test_fe['CGPA'] / (X_test_fe['Attendance_Rate'] + 1e-6)

# --- Step 2.1: Boolean Encoding for 'Gender' ---
X_train_fe['Is_Male'] = (X_train_fe['Gender'] == 'Male') # Now stores True/False
X_test_fe['Is_Male'] = (X_test_fe['Gender'] == 'Male') # Now stores True/False
# Drop the original 'Gender' column as it's now encoded
X_train_fe = X_train_fe.drop(columns=['Gender'])
X_test_fe = X_test_fe.drop(columns=['Gender'])

print("\n--- X_train with new engineered feature ('CGPA_Per_Attendance') and boolean 'Is_Male' ---")
display(X_train_fe[['CGPA', 'Attendance_Rate', 'CGPA_Per_Attendance', 'Is_Male']].head())

# Redefine numerical and categorical columns after feature engineering and boolean encoding
# numerical_cols_final now only includes 'number' types for scaling
numerical_cols_final = X_train_fe.select_dtypes(include=np.number).columns.tolist()
categorical_cols_final = X_train_fe.select_dtypes(include='object').columns.tolist()

# Note: 'Is_Male' (boolean type) is not in numerical_cols_final or categorical_cols_final.
# It will be passed through untouched by the 'remainder="passthrough"' argument in ColumnTransformer.

# --- Step 3 & 4: Encoding and Scaling Pipeline ---

# This creates preprocessing steps for numerical features (scaling)
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()) # Standardize numerical features
])

# This creates preprocessing steps for categorical features (one-hot encoding)
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Convert categorical features to one-hot encoded vectors
])

# This creates a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols_final),
        ('cat', categorical_transformer, categorical_cols_final)
    ],
    remainder='passthrough' # Pass through other columns (like 'Is_Male' boolean) as they are
)

# Apply preprocessing to X_train_fe and X_test_fe
# fit_transform on training data to learn parameters, transform on test data to apply learned parameters
X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)

# Get feature names after one-hot encoding for better interpretability
# preprocessor.get_feature_names_out() will correctly include passed-through columns
feature_names = preprocessor.get_feature_names_out()

# These convert the processed arrays back to DataFrames with proper column names and indices
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train_fe.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test_fe.index)

# Convert one-hot encoded columns (which are currently float64) to boolean True/False
for col in feature_names:
    if col.startswith('cat__'): # Identify columns created by OneHotEncoder
        X_train_processed[col] = X_train_processed[col].astype(bool)
        X_test_processed[col] = X_test_processed[col].astype(bool)

print("\nData Cleaning and Preprocessing Complete! Training and Test sets are ready.")

--- Missing values after imputation in X_train (should be 0) ---
0
--- Missing values after imputation in X_test (should be 0) ---
0

--- X_train with new engineered feature ('CGPA_Per_Attendance') and boolean 'Is_Male' ---


,CGPA,Attendance_Rate,CGPA_Per_Attendance,Is_Male
9252,1.95,65.9,0.029590,True
598,3.62,79.2,0.045707,True
7394,2.47,62.3,0.039647,False
5027,1.39,100.0,0.013900,False
1396,3.98,96.3,0.041329,False



Data Cleaning and Preprocessing Complete! Training and Test sets are ready.


## 6. ***Data After Cleanup (Training Data)***

At this stage, we examined the training data after all the preprocessing steps to ensure it's in the correct format for machine learning algorithms. We expect all columns to be numerical, scaled, and free of missing values.

In [13]:
print("--- X_train Head After Cleanup ---")
display(X_train_processed.head())

print("\n--- X_train Info After Cleanup ---")
X_train_processed.info()

print("\n--- Missing Values in X_train After Cleanup (should be 0) ---")
print(X_train_processed.isnull().sum().sum())

--- X_train Head After Cleanup ---


,num__Age,num__Family_Income,num__Study_Hours_per_Day,num__Attendance_Rate,num__Assignment_Delay_Days,num__Travel_Time_Minutes,num__Stress_Index,num__GPA,num__Semester_GPA,num__CGPA,...,cat__Department_Arts,cat__Department_Business,cat__Department_CS,cat__Department_Engineering,cat__Department_Science,cat__Parental_Education_Bachelor,cat__Parental_Education_High School,cat__Parental_Education_Master,cat__Parental_Education_PhD,remainder__Is_Male
9252,-0.471034,-0.638619,-0.572784,-1.923471,1.637118,0.388561,-0.592911,-0.379287,-0.402407,-0.318107,...,True,False,False,False,False,True,False,False,False,1.0
598,-0.471034,-0.638619,-0.675290,-0.308092,1.637118,-0.704834,-1.577318,1.094498,1.220688,1.232878,...,False,False,False,True,False,False,False,True,False,1.0
7394,0.042598,-0.184928,1.516757,-2.360717,0.155659,-1.243121,1.375903,-0.266641,0.154083,0.164834,...,False,False,False,True,False,False,False,True,False,0.0
5027,-0.471034,0.513969,0.980573,2.218217,0.155659,-0.587084,0.275684,-0.839258,-0.893973,-0.838198,...,True,False,False,False,False,False,False,False,True,0.0
1396,-1.404910,-0.638619,-1.369175,1.768825,-1.325800,-0.671191,-0.071754,1.385500,1.563857,1.567222,...,False,True,False,False,False,False,True,False,False,0.0



--- X_train Info After Cleanup ---
<class 'pandas.core.frame.DataFrame'>
Index: 7000 entries, 9252 to 6816
Data columns (total 31 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   num__Age                             7000 non-null   float64
 1   num__Family_Income                   7000 non-null   float64
 2   num__Study_Hours_per_Day             7000 non-null   float64
 3   num__Attendance_Rate                 7000 non-null   float64
 4   num__Assignment_Delay_Days           7000 non-null   float64
 5   num__Travel_Time_Minutes             7000 non-null   float64
 6   num__Stress_Index                    7000 non-null   float64
 7   num__GPA                             7000 non-null   float64
 8   num__Semester_GPA                    7000 non-null   float64
 9   num__CGPA                            7000 non-null   float64
 10  num__CGPA_Per_Attendance             7000 non-null   float64
 

## 7. ***Model Training and Evaluation***

With the data now cleaned and preprocessed, we can train a baseline machine learning model. We'll use **Logistic Regression** due to its interpretability and effectiveness as a solid baseline for binary classification tasks. We'll evaluate its performance using common metrics like Accuracy, ROC-AUC score, and a detailed classification report.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Initialize and train the Logistic Regression model
# max_iter is increased to ensure convergence for complex datasets
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_processed, y_train)

print("Logistic Regression Model Training Complete!")

# Make predictions on the preprocessed test data
y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1] # Probabilities for the positive class (dropout)

print("\n" + "="*40)
print("        MODEL PERFORMANCE REPORT        ")
print("="*40)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

# 10. Identifying Most Important Features
# Extract feature importances (coefficients for Logistic Regression)
# Take the absolute value as both positive and negative coefficients indicate importance
feature_importance = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Importance': np.abs(model.coef_[0])
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 10 Most Predictive Features (based on absolute coefficients) ---")
display(feature_importance.head(10))

Logistic Regression Model Training Complete!

        MODEL PERFORMANCE REPORT        
Accuracy: 0.8180
ROC-AUC Score: 0.8176

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      2294
           1       0.69      0.41      0.51       706

    accuracy                           0.82      3000
   macro avg       0.77      0.68      0.70      3000
weighted avg       0.80      0.82      0.80      3000


--- Top 10 Most Predictive Features (based on absolute coefficients) ---


,Feature,Importance
7,num__GPA,1.125390
12,cat__Internet_Access_Yes,0.367987
3,num__Attendance_Rate,0.326127
13,cat__Part_Time_Job_No,0.315800
6,num__Stress_Index,0.300567
10,num__CGPA_Per_Attendance,0.258272
28,cat__Parental_Education_Master,0.248136
17,cat__Semester_Year 1,0.222095
4,num__Assignment_Delay_Days,0.218038
15,cat__Scholarship_No,0.216021
